### Pyspark and RDD


Spark Archtiecture  <br>
- It is a lazy Evalution  ---> filter + transformation (Until we call the Action trigger ) Like  reduceBykey ,map,flatmap,mapValues,groupBykey and actions like take(),collect() . Shows Output when we used actions triggers only <br>
- It contains 2 nodes called master and worker nodes , where master node manages the work load , where the worker node do's acutal work. <br>

Driver Node: Coordinates the job, builds the DAG, schedules tasks.

Cluster Manager: Allocates resources (CPU/memory) to executors.

Worker Nodes: Run executors (the actual JVM processes).

Executors: Execute tasks, store data in memory/disk, report results.

Tasks: Smallest unit of work, each processes one partition.

Transformations --> Narrow (non-Shuffle), Wide (Shuffle)

Narrow t/f :

Each input partition maps to exactly one output partition.

No data shuffle across nodes.

Examples: map, filter, mapValues.

Stores locally


Wide t/f :

Input partitions may contribute to multiple output partitions.

Requires shuffle (data exchange across executors).

Examples: groupByKey, reduceByKey, join.

Stores in disk I/O


-------------------------------------------------------------------------------------------------
                                    Driver Node 
                                       |
                                       |
                                    Cluster Manager
                                       |
                                       |
                                    Worker Node 
                                    /         \
                                   /           \
                            Executor Node     Exceutor Node
                                  |               |
                            |   |   |  |      |  |  | |
                          Task1 T2  T3 T4     T1 T2 T3 T4
--------------------------------------------------------------------------------------------------

In [0]:
from pyspark.sql import SparkSession  # Runs only in any compute rather than the serverless
spark=SparkSession.builder.appName("Sparky-Spark").getOrCreate()
sc=spark.sparkContext
data=sc.parallelize([('Alice',10),('Bob',20),('james',30),('Alice',40)])
data.collect() # Actions 

In [0]:
data.filter(lambda x:x[1] >= 20).map(lambda x:x[0]).collect() # Transformations - Narrow

In [0]:
data.groupByKey().mapValues(lambda x:list(x)).collect() # Transformations - Wide 

In [0]:
# Creating a dbfs path ! 
dbutils.fs.mkdirs('/FileStore/csv_files/')


In [0]:
display(dbutils.fs.ls('/FileStore/csv_files/ord.csv'))

In [0]:
# Using csv for rdd
rdd_data=sc.textFile('/FileStore/csv_files/ord.csv')
rdd_data.take(3)

In [0]:
%sql
use catalog starboy_9;
create schema db_j;
use schema db_j ;
create volume v_files;

In [0]:
# dbutils.fs.cp('/FileStore/csv_files/ord.csv','/Volumes/starboy_9/db_j/v_files/ord_v.csv')
dbutils.fs.ls('/Volumes/starboy_9/db_j/v_files/ord_v.csv')

In [0]:
csv_data=spark.read.text('/Volumes/starboy_9/db_j/v_files/ord_v.csv').rdd
raw_data=sc.parallelize(csv_data.collect())

In [0]:
raw_data.collect()

### Pyspark Df and Sql 

In [0]:
df=spark.read.csv('/Volumes/starboy_9/db_j/v_files/ord_v.csv',header=True,inferSchema=True)
# display(df)
df.show()

In [0]:
# pyspark ops
from pyspark.sql.functions import *
res1=df.groupBy('payment_method').agg(sum('order_amount').alias('Total_amount')).orderBy('Total_amount',ascending=False).limit(2)
display(res1)

In [0]:
df.withColumn('order_amount', col('order_amount').cast('float')).show()
df.printSchema()

In [0]:
df_join=df.join(res1,on="payment_method",how='left')
df_join.show()

In [0]:
df.createOrReplaceTempView('sample_df')
display(spark.sql('select * from sample_df'))

### PySpark Streaming 


In [0]:

from pyspark.sql.types import *
man_schema=StructType([
    StructField('order_id',IntegerType(),True),
    StructField('customer_id',StringType(),True),
    StructField('order_date',DateType(),True),
    StructField('shipping_city',StringType(),True),
    StructField('payment_method',StringType(),True),
    StructField('order_status',StringType(),True),
    StructField('order_amount',IntegerType(),True)])

In [0]:
spark.unpersist()

In [0]:
# structure Streaming without Autoloader 
stream=(spark.readStream.schema(man_schema).csv('/Volumes/starboy_9/db_j/v_files/',header=True)
        .writeStream
        .format('delta')
        .option('checkpointLocation','/Volumes/starboy_9/db_j/v_files/chk_pt')
        .toTable('orders_v'))

In [0]:
%sql
create volume str_v 

In [0]:
# AutoLoader Streaming !!
stream_auto=(spark.readStream.format('cloudFiles')
        .option('cloudFiles.format','csv')
        .option('cloudFiles.schemaLocation','/Volumes/starboy_9/db_j/v_files/schema_chk')
        .option('header','true')
        .option('mergeSchema','true')
        .load('/Volumes/starboy_9/db_j/v_files/csv/'))

(stream_auto.writeStream.format('delta').option('checkpointLocation','/Volumes/starboy_9/db_j/v_files/schema_chk').outputMode('append')\
            .toTable('starboy_9.db_j.orders_v'))

In [0]:
dbutils.fs.rm("/Volumes/starboy_9/db_j/v_files/schema_chk/",recurse=True)
dbutils.fs.rm("/Volumes/starboy_9/db_j/v_files/chk/",recurse=True)
spark.sql("drop table if exists orders_v")

In [0]:
%sql
select * from starboy_9.db_j.orders_v 

In [0]:
spark.table('starboy_9.db_j.orders_v ').write.save('/Volumes/starboy_9/db_j/v_files/tables_metadata')

In [0]:
display(dbutils.fs.ls('/Volumes/starboy_9/db_j/v_files/tables_metadata'))

### Databricks Sql 

- Catalog
- Schema
- Volumes 
- Tables 
- Views 

#### SQL - reading 

In [0]:
%sql
create or replace table starboy_9.db_j.t_csv 
as 
select * from csv.`/Volumes/starboy_9/db_j/v_files/csv/ord1.csv` with (header='true',inferSchema='true');
select * from starboy_9.db_j.t_csv

In [0]:
%sql
create or replace table starboy_9.db_j.t_csv1
as 
select * from read_files('/Volumes/starboy_9/db_j/v_files/csv/ord1.csv',format => 'csv',header =>'true',inferSchema=>'true') ;
select * from starboy_9.db_j.t_csv1

In [0]:
%sql
create or replace table starboy_9.db_j.t_csv2
as 
select * from cloud_files('/Volumes/starboy_9/db_j/v_files/csv/', 'csv', map('header', 'true', 'inferSchema', 'true'));
select * from starboy_9.db_j.t_csv2;

In [0]:
%sql
select * from starboy_9.db_j.t_csv

#### External Location

In [0]:
dbutils.secrets.listScopes()

In [0]:
st=dbutils.secrets.get(scope='azure',key='storage-key')
spark.conf.set('fs.azure.account.key.kishoredbstorage.dfs.core.windows.net',st)

In [0]:
display(dbutils.fs.ls('abfss://class@kishoredbstorage.dfs.core.windows.net/csv/'))

In [0]:
storage='abfss://class@kishoredbstorage.dfs.core.windows.net/csv'
df_del=spark.read.csv(f'{storage}/food_delivery.csv',header=True,inferSchema=True)
display(df_del)

In [0]:
df_del.write.format('delta').save('abfss://class@kishoredbstorage.dfs.core.windows.net/t_m/')

In [0]:
df = spark.read.format("delta").load("abfss://class@kishoredbstorage.dfs.core.windows.net/t_m")
df.show()


#### Internal Location 

In [0]:
df_del.write.format('delta').save('/Volumes/starboy_9/db_j/str_v/table_metdata')

In [0]:
df1=spark.read.format('delta').load('/Volumes/starboy_9/db_j/str_v/table_metdata/')
df1.show()

In [0]:
df_del.groupBy('DeliveryStatus').count().write.format('delta').saveAsTable('starboy_9.db_j.del_status_count')

In [0]:
%sql
select * from starboy_9.db_j.del_status_count

#### Partition By and Repartition

In [0]:
## Partitions 
df_del.write.format('delta').partitionBy('DeliveryStatus').save('/Volumes/starboy_9/db_j/str_v/partition_table/')

In [0]:
df_del.rdd.getNumPartitions()

In [0]:
display(dbutils.fs.ls('/Volumes/starboy_9/db_j/str_v/partition_table/'))

In [0]:
# Repartitioning
df_reloaded = spark.read.format("delta").load("/Volumes/starboy_9/db_j/str_v/partition_table/")
df_reloaded.show()

#### Delta logs  && Delta transactions 

In [0]:
display(dbutils.fs.ls('/Volumes/starboy_9/db_j/str_v/table_metdata/_delta_log/'))

In [0]:
df_json=spark.read.json('/Volumes/starboy_9/db_j/str_v/table_metdata/_delta_log/00000000000000000000.json')
display(df_json)

In [0]:
%sql
describe history starboy_9.db_j.orders_v;


In [0]:
%sql
restore starboy_9.db_j.orders_v version as of 4;

In [0]:
%sql
describe detail starboy_9.db_j.orders_v 

In [0]:
%sql
optimize starboy_9.db_j.orders_v

In [0]:
%sql
describe detail starboy_9.db_j.orders_v ;

In [0]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
# To override the default value of 7 days

In [0]:
%sql
vacuum starboy_9.db_j.orders_v retain 0 hours dry run

In [0]:
%sql
vacuum starboy_9.db_j.orders_v 

In [0]:
%sql
desc history starboy_9.db_j.orders_v 

In [0]:
%sql
OPTIMIZE starboy_9.db_j.orders_v
ZORDER BY (order_id);


#### Views,Temp Views ,CTE ,subquery

In [0]:
%sql
create or replace temp view temp_view as 
select * from starboy_9.db_j.orders_v;
select * from temp_view;

In [0]:
%sql
select round(avg(total_revenue),2) as avg_rev from (select order_id,sum(cast(quantity as DECIMAL(20,2))*cast(unit_price as DECIMAL(20,2))) as total_revenue from starboy_9.db_j.orders_v where order_status='COMPLETED' group by order_id)

In [0]:
%sql
with cte as (select order_id,sum(cast(quantity as DECIMAL(20,2))*cast(unit_price as DECIMAL(20,2))) as total_revenue from starboy_9.db_j.orders_v where order_status='COMPLETED' group by order_id)
select round(avg(total_revenue),2) as avg_rev from cte

#### Merge and copy into 

In [0]:
%sql
create table cafe_od

In [0]:
%sql
copy into cafe_od 
from '/Volumes/starboy_9/db_j/v_files/csv/'
fileformat = csv
format_options('header'='true','inferSchema'='true')
copy_options('mergeSchema'='true')

In [0]:
%sql
select count(*) from cafe_od --Before

In [0]:
%sql
select count(*) from cafe_od  -- After adding files and manual ran 

In [0]:
%sql
-- Filtering 
create or replace table filter_del 
as 
select * from orders_v 
where city ='Kakinada';
select * from filter_del;

In [0]:
%sql
merge into filter_del as t
using orders_v as s 
on t.order_id=s.order_id
when matched then update set t.order_status=s.order_status , t.quantity=s.quantity
when not matched then insert *

In [0]:
%sql
update orders_v set order_status='CANCELLED' where order_id ='O2001'

In [0]:
%sql
--Before the merge into !!
select order_id,order_status from filter_del where order_id='O2001'

In [0]:
%sql
--After the merge into !!
select order_id,order_status from filter_del where order_id='O2001'

In [0]:
%sql
-- O2003	C203	P603	Grocery	5	500	2026-06-12	Kakinada	CANCELLED	null
insert into orders_v values('O2016','C220','P999','Electronics',6,600,'2026-06-12','Kakinada','COMPLETED','null')

In [0]:
%sql
--Before
select * from filter_del where order_id='O2016'

In [0]:
%sql
--After
select * from filter_del where order_id='O2016'

In [0]:
%sql
alter table orders_v 
add columns (total_revenue decimal(20,2));
    
update orders_v set total_revenue=cast(quantity as DECIMAL(20,2))*cast(unit_price as DECIMAL(20,2));

In [0]:
%sql
select * from orders_v


In [0]:
%sql
alter table filter_del add column (total_revenue decimal(20,2))

In [0]:
%sql
merge into filter_del as t
using orders_v as s 
on t.order_id=s.order_id
when matched and s.order_status in ('CANCELLED','PENDING') then delete
when matched then update set t.order_status=s.order_status , t.quantity=s.quantity,t.total_revenue=s.total_revenue
when not matched then insert *

In [0]:
%sql
--Before
select * from filter_del ;

In [0]:
%sql
--After
select * from filter_del;
-- See rows are deleted and got new col values 

#### Multi-Hop architecture 


In [0]:
%sql
use catalog starboy_9;
create schema m_hop ;
use schema m_hop;
create volume v_files ;

In [0]:

display(dbutils.fs.ls('/Volumes/starboy_9/m_hop/v_files/data_files/'))

## **Note** 

This series part-1 is now Ended !!!! 

---------------------------------------------------------------------------------------------------------------------
Let's Pray for God's Like 'Doctor Manju - The confuser'  <br> and <br> 'Professor Gopal Karthik - The Punisher'
-----------------------------------------------------------------------------------------------------------------------

This series is now shifted to part-2